# أثر × ALLaM — Hail heritage test

Does **ALLaM-7B** (SDAIA / HUMAIN, Apache-2.0) know Hail's heritage? This asks it 24 questions built from
Athr's own verified site facts (`src/data/sites.js`), scores the answers, then asks 3 open "tour guide" questions.

**Kaggle setup (once):** ⚙️ *Settings* → **Accelerator: GPU T4 x2** · **Internet: On**. Then *Run All*.
First run downloads ~15 GB (≈5–10 min). Results land in `/kaggle/working/` — download them from the *Output* tab.

In [ ]:
!pip -q install -U "transformers>=4.40.1" accelerate

In [ ]:
import torch, json, re, time
from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL = "humain-ai/ALLaM-7B-Instruct-preview"
tok = AutoTokenizer.from_pretrained(MODEL)
model = AutoModelForCausalLM.from_pretrained(MODEL, torch_dtype=torch.float16, device_map="auto")  # T4 = fp16
model.eval()
print("loaded on", {str(p.device) for p in model.parameters()})

In [ ]:
def ask(question, max_new_tokens=200):
    msgs = [{"role": "user", "content": question}]
    text = tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
    inputs = tok(text, return_tensors="pt", return_token_type_ids=False).to(model.device)
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)  # greedy = repeatable
    return tok.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()

print(ask("كيف أجهز كوب شاهي؟"))  # the model card's own smoke test

In [ ]:
EVAL = {
 "sites": [
  {
   "id": "jubbah",
   "name": "نقوش جبة الصخرية"
  },
  {
   "id": "aarif",
   "name": "قلعة أَعَيْرِف"
  },
  {
   "id": "qishlah",
   "name": "قصر القشلة"
  },
  {
   "id": "aja",
   "name": "جبال أجا"
  },
  {
   "id": "museum",
   "name": "متحف حائل الإقليمي"
  },
  {
   "id": "shuwaymis",
   "name": "نقوش الشويمس"
  }
 ],
 "questions": [
  {
   "site": "نقوش جبة الصخرية",
   "q": "ما الإدراج في اليونسكو لموقع «نقوش جبة الصخرية» في منطقة حائل؟ أجب باختصار.",
   "expected": "2015 م"
  },
  {
   "site": "نقوش جبة الصخرية",
   "q": "ما عدد اللوحات التقديري لموقع «نقوش جبة الصخرية» في منطقة حائل؟ أجب باختصار.",
   "expected": "أكثر من 5,400 نقش"
  },
  {
   "site": "نقوش جبة الصخرية",
   "q": "ما أقدم النقوش لموقع «نقوش جبة الصخرية» في منطقة حائل؟ أجب باختصار.",
   "expected": "نحو 10,000 سنة"
  },
  {
   "site": "نقوش جبة الصخرية",
   "q": "ما المسافة من حائل لموقع «نقوش جبة الصخرية» في منطقة حائل؟ أجب باختصار.",
   "expected": "95 كم شمال غربًا"
  },
  {
   "site": "قلعة أَعَيْرِف",
   "q": "ما العمر التقديري لموقع «قلعة أَعَيْرِف» في منطقة حائل؟ أجب باختصار.",
   "expected": "أكثر من 200 عام"
  },
  {
   "site": "قلعة أَعَيْرِف",
   "q": "ما الارتفاع عن المدينة لموقع «قلعة أَعَيْرِف» في منطقة حائل؟ أجب باختصار.",
   "expected": "نحو 30 مترًا"
  },
  {
   "site": "قلعة أَعَيْرِف",
   "q": "ما الصعود لموقع «قلعة أَعَيْرِف» في منطقة حائل؟ أجب باختصار.",
   "expected": "درجٌ فقط · لا طريق للسيارة"
  },
  {
   "site": "قلعة أَعَيْرِف",
   "q": "ما مادة البناء لموقع «قلعة أَعَيْرِف» في منطقة حائل؟ أجب باختصار.",
   "expected": "طين وتبن وجذوع أثل"
  },
  {
   "site": "قصر القشلة",
   "q": "ما عدد الغرف لموقع «قصر القشلة» في منطقة حائل؟ أجب باختصار.",
   "expected": "142 غرفة"
  },
  {
   "site": "قصر القشلة",
   "q": "ما أبراج المراقبة لموقع «قصر القشلة» في منطقة حائل؟ أجب باختصار.",
   "expected": "8 أبراج"
  },
  {
   "site": "قصر القشلة",
   "q": "ما سنة البناء لموقع «قصر القشلة» في منطقة حائل؟ أجب باختصار.",
   "expected": "1360هـ / 1941م"
  },
  {
   "site": "قصر القشلة",
   "q": "ما مادة البناء لموقع «قصر القشلة» في منطقة حائل؟ أجب باختصار.",
   "expected": "طين وتبن وجذوع أثل"
  },
  {
   "site": "جبال أجا",
   "q": "ما أعلى قمة لموقع «جبال أجا» في منطقة حائل؟ أجب باختصار.",
   "expected": "نحو 1,500 م فوق سطح البحر"
  },
  {
   "site": "جبال أجا",
   "q": "ما نوع الصخر لموقع «جبال أجا» في منطقة حائل؟ أجب باختصار.",
   "expected": "جرانيت وردي وبركانيات قديمة"
  },
  {
   "site": "جبال أجا",
   "q": "ما الامتداد لموقع «جبال أجا» في منطقة حائل؟ أجب باختصار.",
   "expected": "أكثر من 100 كم"
  },
  {
   "site": "جبال أجا",
   "q": "ما الارتباط التاريخي لموقع «جبال أجا» في منطقة حائل؟ أجب باختصار.",
   "expected": "قبيلة طيء · حاتم الطائي"
  },
  {
   "site": "متحف حائل الإقليمي",
   "q": "ما أقسام العرض لموقع «متحف حائل الإقليمي» في منطقة حائل؟ أجب باختصار.",
   "expected": "ما قبل التاريخ · الممالك · الإسلامي · التراث الشعبي"
  },
  {
   "site": "متحف حائل الإقليمي",
   "q": "ما مدة الزيارة المثالية لموقع «متحف حائل الإقليمي» في منطقة حائل؟ أجب باختصار.",
   "expected": "90 دقيقة"
  },
  {
   "site": "متحف حائل الإقليمي",
   "q": "ما مناسب للأطفال لموقع «متحف حائل الإقليمي» في منطقة حائل؟ أجب باختصار.",
   "expected": "نعم — قاعات مكيّفة ومسار قصير"
  },
  {
   "site": "متحف حائل الإقليمي",
   "q": "ما الدخول لموقع «متحف حائل الإقليمي» في منطقة حائل؟ أجب باختصار.",
   "expected": "مجاني"
  },
  {
   "site": "نقوش الشويمس",
   "q": "ما الإدراج في اليونسكو لموقع «نقوش الشويمس» في منطقة حائل؟ أجب باختصار.",
   "expected": "2015 م (مع جبة)"
  },
  {
   "site": "نقوش الشويمس",
   "q": "ما المكوّنان لموقع «نقوش الشويمس» في منطقة حائل؟ أجب باختصار.",
   "expected": "جبل المنجور · جبل راط"
  },
  {
   "site": "نقوش الشويمس",
   "q": "ما المسافة من حائل لموقع «نقوش الشويمس» في منطقة حائل؟ أجب باختصار.",
   "expected": "250 كم جنوب غربًا"
  },
  {
   "site": "نقوش الشويمس",
   "q": "ما الوصول لموقع «نقوش الشويمس» في منطقة حائل؟ أجب باختصار.",
   "expected": "مرشد معتمد · جزء من الطريق ترابي"
  }
 ]
}

In [ ]:
AR_DIGITS = str.maketrans("٠١٢٣٤٥٦٧٨٩", "0123456789")

def numbers(s):
    s = s.translate(AR_DIGITS).replace(",", "").replace("٬", "")
    return re.findall(r"\d+(?:\.\d+)?", s)

def score(expected, answer):
    """✅ every number in the expected fact is in the answer · ❌ one is missing · 👀 no numbers: judge by eye"""
    exp = numbers(expected)
    if not exp:
        return "👀"
    got = set(numbers(answer))
    return "✅" if all(n in got for n in exp) else "❌"

rows = []
for i, item in enumerate(EVAL["questions"], 1):
    t = time.time()
    a = ask(item["q"])
    rows.append({**item, "answer": a, "score": score(item["expected"], a), "secs": round(time.time() - t, 1)})
    print(f"{i:2}/{len(EVAL['questions'])} {rows[-1]['score']} {item['site']} | expected: {item['expected']}")
    print(f"     ALLaM: {a[:160]}\n")

In [ ]:
graded = [r for r in rows if r["score"] != "👀"]
right = sum(r["score"] == "✅" for r in graded)
print(f"Numeric facts correct: {right}/{len(graded)}  ({100 * right / max(1, len(graded)):.0f}%)")
print(f"To judge by eye: {sum(r['score'] == '👀' for r in rows)}")
by_site = {}
for r in graded:
    by_site.setdefault(r["site"], []).append(r["score"] == "✅")
for site, ok in by_site.items():
    print(f"  {site}: {sum(ok)}/{len(ok)}")

## Open questions — would ALLaM make a good tour guide?

In [ ]:
OPEN = [
    "أنا سائح أزور حائل لأول مرة. اشرح لي نقوش جبة الصخرية في خمس جمل بأسلوب مرشد سياحي.",
    "ما الفرق بين النقوش الثمودية والنبطية؟ وأين أجدها في منطقة حائل؟",
    "خطط لي يومًا واحدًا في حائل يجمع التاريخ والطبيعة، مع أفضل وقت لكل محطة.",
]
open_answers = []
for q in OPEN:
    a = ask(q, max_new_tokens=500)
    open_answers.append({"q": q, "answer": a})
    print("Q:", q)
    print(a)
    print("-" * 60)

In [ ]:
with open("/kaggle/working/allam_athr_results.json", "w", encoding="utf-8") as f:
    json.dump({"model": MODEL, "facts": rows, "open": open_answers}, f, ensure_ascii=False, indent=1)
lines = ["| | Site | Expected | ALLaM |", "|---|---|---|---|"]
for r in rows:
    ans = r["answer"][:120].replace("|", "/").replace("\n", " ")
    lines.append(f"| {r['score']} | {r['site']} | {r['expected']} | {ans} |")
with open("/kaggle/working/allam_athr_results.md", "w", encoding="utf-8") as f:
    f.write("\n".join(lines))
print("Saved -> Output tab: allam_athr_results.json + .md")